# pipetree on Microsoft Fabric

Runs the same example pipeline as `examples/run_demo.py`, against a Fabric lakehouse instead of a laptop.

**Status: not verified against a real workspace yet.** Built and unit-tested against a mocked `notebookutils` (`tests/platform/test_fabric.py`), but nothing here has actually run in Fabric. See `examples/databricks/README.md`'s sibling notes and `NOTES-for-blog.md` for the same caveat applied here: `notebookutils.credentials.getSecret` and the `Files/` mount convention are documented Fabric APIs, but the exact shape hasn't been checked against a live workspace.

Before running: upload `examples/pipeline.yaml`, `examples/data/`, and `examples/notebooks/` into this lakehouse's `Files/` section (matching the paths the config expects), and install the `pipetree` wheel - either via a Fabric **environment** with it as a custom library (recommended), or with `%pip install` in the cell below pointing at wherever you uploaded the `.whl`.

In [ ]:
# Only needed if pipetree isn't already installed via a Fabric environment.
# %pip install /lakehouse/default/Files/pipetree-0.1.0-py3-none-any.whl --quiet

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession

from pipetree import run_pipeline
from pipetree.adapters.spark.adapter import SparkAdapter
from pipetree.config.loader import load_config
from pipetree.model import PipelineConfig
from pipetree.platform.fabric import FabricPlatform

# unused by the example pipeline (no secrets); set this for a real config
KEY_VAULT_URL = "https://<your-vault>.vault.azure.net/"
CONFIG_PATH = Path("/lakehouse/default/Files/pipeline.yaml")

raw = load_config(CONFIG_PATH)
config = PipelineConfig.from_validated_raw(raw)

spark = SparkSession.getActiveSession()
assert spark is not None, "expected an active SparkSession - run this in Fabric"

# notebookutils is injected by the Fabric runtime as a global; not
# importable, so this cell only works inside an actual Fabric notebook.
platform = FabricPlatform(notebookutils, key_vault_url=KEY_VAULT_URL)  # noqa: F821
adapter = SparkAdapter(
    spark, systems=config.systems, base_dir=CONFIG_PATH.parent, platform=platform
)

digest = run_pipeline(CONFIG_PATH, adapter=adapter)
digest.exit_code